In [1]:
# import libraries
import os, sys
import matplotlib.pyplot as plt

# --- Configuration ---
from ICESEE.config._utility_imports import *
from ICESEE.src.utils.tools import icesee_savefig

[ICESEE] Running in Jupyter - disabling command line arguments


In [2]:
data_dir='_modelrun_datasets'

### Loading and figures: `ICESEE.src.postprocessing`
The outputs are read and the example's result figures are drawn by
`ICESEE.src.postprocessing` (the same code the ICESEE launcher's Results
view uses; see `results.yaml`). The cells below keep the notebook's variable
names, so the exploratory cells further down work unchanged.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import ICESEE
from ICESEE.src.postprocessing import load_contract
from ICESEE.src.postprocessing.render import open_context, render_view

# results.yaml (next to this notebook) says where the outputs are and which views exist
example_dir = Path(ICESEE.__file__).parent / "applications/lorenz_model/examples/lorenz96"
contract = load_contract(Path.cwd()) or load_contract(example_dir)

def show(view, png, **options):
    """Draw one result view (ICESEE.src.postprocessing.views) and save it as png."""
    rendered, fig = render_view(ctx, view, options or None, png=Path(png), dpi=300)
    for note in rendered.notes:
        print(note)
    if rendered.table:
        print("  ".join(f"{c:>12}" for c in rendered.table["columns"]))
        for row in rendered.table["rows"]:
            print("  ".join(f"{c:>12}" for c in row))
    if fig is not None:
        display(Image(filename=str(png)))
    return rendered

filter_type = "true-wrong"
model_name = "lorenz"
ctx = open_context(data_dir, contract, filter_type="EnKF", params=None)
out = ctx.outputs                      # ICESEE.src.postprocessing.outputs.DAOutputs
t, ind_m, tm_m, run_mode = out.t, out.obs_index, out.obs_max_time, out.run_mode
ensemble_true_state, ensemble_nurged_state = out.true_state, out.nurged_state
ensemble_vec_mean, ensemble_bg = out.ensemble_mean, out.ensemble_bg
w = out.observations
datasets_tw = {"t": t, "obs_index": ind_m, "obs_max_time": np.array([tm_m]), "run_mode": np.array([run_mode]),
               **out.metadata}
if out.ensemble_file is not None:   # full ensemble, for the exploratory cells below
    with h5py.File(out.ensemble_file, "r") as f:
        ensemble_vec_full = f[out.ensemble_key][:]
for warning in out.warnings:
    print("warning:", warning)
t.shape

In [5]:
ensemble_true_state.shape
# t.shape
# 10/0.01

(3, 1001)

In [ ]:
# true, background, observations and analysis for x, y, z (observation window shaded)
figure_dir = Path("_modelrun_datasets") / "figures"
show("lorenz.state_estimation", figure_dir / "results.png")

In [7]:
all_flters = False
if all_flters:
    filter_types = ["EnKF", "DEnKF", "EnTKF", "EnRSKF"]

    # Prepare data for dynamic subplot layout
    available_filters = []
    datasets_dict = {}

    for filter_type in filter_types:
        file_path = f"_modelrun_datasets/{filter_type}-{model_name}.h5"
        try:
            datasets = extract_datasets_from_h5(file_path)
            datasets_dict[filter_type] = datasets
            available_filters.append(filter_type)  # Keep track of filters with data
        except FileNotFoundError:
            print(f"Warning: Dataset for {filter_type} not found. Skipping.")
        except KeyError as e:
            print(f"Warning: Missing dataset key {e} in {filter_type}. Skipping.")

    # Dynamic layout based on available filters
    ncols = len(available_filters)
    if ncols == 0:
        raise ValueError("No datasets available for any filter type!")

    # Dynamically adjust figsize to maintain aspect ratio
    base_width = 5
    base_height = 3  # Base height per subplot row
    fig_width = max(base_width * ncols, 10)  # Ensure minimum width for clarity
    fig_height = base_height * 3  # 3 rows for x(t), y(t), z(t)

    fig, axs = plt.subplots(nrows=3, ncols=ncols, figsize=(fig_width, fig_height), sharex=True)

    # If there's only one column, ensure axs is 2D
    if ncols == 1:
        axs = np.expand_dims(axs, axis=1)

    # Colors and line styles
    styles = {
        "True": {"color": "blue", "linewidth": 2, "label": r'\bf{True}'},
        "Background": {"color": "orange", "linestyle": ":", "linewidth": 2, "label": r'\bf{Background}'},
        "Observation": {"color": "green", "marker": "o", "linestyle": "none", "fillstyle": "none", "markersize": 6, "markeredgewidth": 1.5, "label": r'\bf{Observation}'},
        "Analysis": {"color": "red", "linestyle": "--", "linewidth": 2, "label": r'\bf{Analysis}'},
    }

    # Define the y-axis limits for each variable
    y_limits = [(-20, 25), (-25, 30), (-5, 50)]  # Adjust these to reflect realistic bounds

    # Plot the available datasets
    for i, filter_type in enumerate(available_filters):
        datasets = datasets_dict[filter_type]

        # Extract datasets based on filter type
        statevec_ens_full = datasets["statevec_ens_full"]
        statevec_ens_mean = datasets["statevec_ens_mean"]
        statevec_bg = datasets["statevec_bg"]

        # Plot each variable (x, y, z) in separate rows
        for k in range(3):
            ax = axs[k, i]
            ax.plot(t, statevec_true[k, :], **styles["True"])
            ax.plot(t, statevec_bg[k, :], **styles["Background"])
            ax.plot(t[ind_m], w[k, :], **styles["Observation"])
            ax.plot(t, statevec_ens_mean[k, :], **styles["Analysis"])

            if k == 2:  # Only set x-axis labels for the bottom row
                ax.set_xlabel(r'$t$', fontsize=16)
            if i == 0:  # Only set y-axis labels for the first column
                ax.set_ylabel([r'$x(t)$', r'$y(t)$', r'$z(t)$'][k], fontsize=16)

            # Set independent y-axis limits
            ax.set_ylim(y_limits[k])

            # Highlight a region of interest
            ax.axvspan(0, tm_m, color='y', alpha=0.2, lw=0)

            # Add the filter type as the title for each column
            if k == 0:
                ax.set_title(filter_type, fontsize=18, weight='bold')

            # Add gridlines
            ax.grid(True, linestyle="--", alpha=0.5)

    # Add a single legend inside the plot grid
    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=4, fontsize=14)

    # Adjust subplot spacing
    fig.subplots_adjust(hspace=0.3, wspace=0.3)

    # Automatically adjust layout to minimize whitespace
    plt.tight_layout()

    # Show the plot
    plt.show()
    